# 📘 KỸ THUẬT LẬP TRÌNH C++ — CHƯƠNG VII & VIII
## Nạp Chồng (Overloading) & Friend trong C++

> **Môn:** Kỹ thuật Lập trình C++ | **Chuẩn:** C++17
> **Cách dùng:** Chạy theo thứ tự từ trên xuống.
> Mỗi ví dụ = **Cell 1** ghi file `.cpp` + **Cell 2** biên dịch & chạy.

---
## 📋 MỤC LỤC

| Phần | Nội dung |
|:----:|----------|
| **VII.1** | Function Overloading |
| **VII.2** | Constructor Overloading |
| **VII.3** | Operator Overloading — Tổng quan |
| **VII.4** | Toán tử số học (+, -, *, /) |
| **VII.5** | Toán tử so sánh (==, !=, <, >, ...) |
| **VII.6** | Toán tử nhập xuất (<<, >>) |
| **VII.7** | Toán tử gán (=) |
| **VII.8** | Toán tử tăng giảm (++, --) |
| **VII.9** | Quy tắc nạp chồng toán tử |
| **VIII.1** | Friend Function |
| **VIII.2** | Friend Class |
| **VIII.3** | Ứng dụng Friend |
| **VIII.4** | Ưu nhược điểm của Friend |
| **★** | Bài tổng hợp: Class PhanSo (Phân Số) |


## ⚙️ SETUP — Cấu hình chạy C++ trên Google Colab

```
Bước 1 — Cell %%writefile:   Ghi code vào file .cpp
Bước 2 — Cell !g++:          Biên dịch và chạy ngay
```


In [ ]:
!g++ --version | head -1
print("✅ Môi trường C++ sẵn sàng!")

---
# VII. NẠP CHỒNG (OVERLOADING)

> **Overloading** = Cùng một **tên**, nhưng hành vi khác nhau tùy **hoàn cảnh**.
> Đây là **đa hình tĩnh (static polymorphism)** — giải quyết tại **compile time**.

```
               ┌─────────────────────────────────────────┐
               │  Overloading — phân loại                │
               │                                         │
               │  Function Overloading                   │
               │    → cùng tên hàm, khác tham số         │
               │                                         │
               │  Operator Overloading                   │
               │    → định nghĩa lại toán tử cho class   │
               └─────────────────────────────────────────┘
```


## VII.1 — Function Overloading (Nạp chồng hàm)

### 🧠 Bản chất — Compiler phân biệt qua "Function Signature"
Compiler nhìn vào **chữ ký hàm** để chọn hàm nào sẽ gọi:

| Yếu tố | Phân biệt được? |
|--------|:-:|
| Số lượng tham số | ✅ |
| Kiểu dữ liệu tham số | ✅ |
| Thứ tự các kiểu tham số | ✅ |
| **Kiểu trả về (return type)** | ❌ **KHÔNG!** |

> ⚠️ **Lỗi cực kỳ phổ biến:** Return type KHÔNG phân biệt overload!
> ```cpp
> int  tinh(int a) { return a; }     // OK
> void tinh(int a) { }               // ❌ LỖI — trùng signature!
> ```

### 🔧 Cơ chế: Name Mangling
Thực chất compiler **đổi tên** hàm thành tên nội bộ duy nhất:
```
in(int)    →  _Z2ini
in(double) →  _Z2ind
in(string) →  _Z2inSs
```
Đây là lý do C++ có overloading còn C thì không!

### 📌 Nguyên tắc thực chiến
- ✅ Dùng khi các hàm thực hiện **cùng một tác vụ** nhưng với kiểu khác
- ❌ KHÔNG dùng cho các hàm có **logic khác nhau** (dù tên giống)
- 💡 **Default argument** đôi khi gọn hơn overloading


In [ ]:
%%writefile vii1_func_overload.cpp
#include <iostream>
#include <cmath>
#include <string>
using namespace std;

// ─── Ví dụ 1: Tính diện tích — cùng tên, khác số tham số ───
double dien_tich(double r) {                          // hình tròn
    return M_PI * r * r;
}
double dien_tich(double dai, double rong) {           // hình chữ nhật
    return dai * rong;
}
double dien_tich(double a, double b, double c) {      // tam giác (Heron)
    double s = (a + b + c) / 2;
    return sqrt(s*(s-a)*(s-b)*(s-c));
}

// ─── Ví dụ 2: In thông tin — cùng tên, khác kiểu tham số ───
void in(int    x) { cout << "[int]    " << x << "\n"; }
void in(double x) { cout << "[double] " << x << "\n"; }
void in(string s) { cout << "[string] " << s << "\n"; }
void in(int x, int y) { cout << "[pair]   (" << x << "," << y << ")\n"; }

// ─── Ví dụ 3: Default argument thay thế overloading ───
void chao(string ten, string loi = "Xin chào") {
    cout << loi << ", " << ten << "!\n";
}

int main() {
    cout << "=== Diện tích ===\n";
    cout << "Hình tròn r=5:        " << dien_tich(5.0) << "\n";
    cout << "HCN 4x6:              " << dien_tich(4.0, 6.0) << "\n";
    cout << "Tam giác 3,4,5:       " << dien_tich(3.0, 4.0, 5.0) << "\n";

    cout << "\n=== In nhiều kiểu ===\n";
    in(42);
    in(3.14);
    in(string("C++ Overloading"));
    in(10, 20);

    cout << "\n=== Default argument ===\n";
    chao("An");
    chao("Bình", "Chào buổi sáng");
    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii1_out vii1_func_overload.cpp && ./vii1_out

## VII.2 — Constructor Overloading (Nạp chồng hàm tạo)

### 🧠 Bản chất
Constructor overloading là **trường hợp đặc biệt** của function overloading.
Cho phép tạo đối tượng theo nhiều cách khác nhau.

### 📋 Các loại constructor phổ biến

| Loại | Cú pháp | Khi nào dùng |
|------|---------|-------------|
| Default | `Foo()` | Tạo đối tượng không có thông tin ban đầu |
| Parameterized | `Foo(int x, int y)` | Tạo với dữ liệu cụ thể |
| Copy | `Foo(const Foo& other)` | Sao chép từ đối tượng khác |
| Delegating (C++11) | `Foo() : Foo(0, 0) {}` | Gọi constructor khác để tránh lặp code |

### 🔥 Delegating Constructor — Kỹ thuật giảm lặp code (C++11)
```cpp
class Hop {
    Hop() : Hop(1, 1, 1) {}         // delegate → Hop(int,int,int)
    Hop(int canh) : Hop(canh,canh,canh) {}  // delegate → Hop(int,int,int)
    Hop(int d, int r, int c) {      // Constructor THỰC SỰ làm việc
        // code khởi tạo ở đây
    }
};
```
> 💡 Mọi logic khởi tạo tập trung vào 1 chỗ → dễ maintain!


In [ ]:
%%writefile vii2_ctor_overload.cpp
#include <iostream>
#include <string>
using namespace std;

class HopChu {
    int dai, rong, cao;

    // Helper in thông tin
    string info() const {
        return to_string(dai)+"×"+to_string(rong)+"×"+to_string(cao);
    }

public:
    // ── Default: hộp 1×1×1 ──
    HopChu() : HopChu(1, 1, 1) {           // Delegating!
        cout << "[Default]      Hộp " << info() << "\n";
    }

    // ── Lập phương: hộp n×n×n ──
    explicit HopChu(int canh) : HopChu(canh, canh, canh) {  // Delegating!
        cout << "[Lập phương]   Hộp " << info() << "\n";
    }

    // ── Parameterized: hộp d×r×c — constructor THỰC SỰ ──
    HopChu(int d, int r, int c) : dai(d), rong(r), cao(c) {
        cout << "[Parameterized] Hộp " << info() << "\n";
    }

    // ── Copy constructor ──
    HopChu(const HopChu& other) : dai(other.dai), rong(other.rong), cao(other.cao) {
        cout << "[Copy]          Hộp " << info() << "\n";
    }

    int the_tich() const { return dai * rong * cao; }
};

int main() {
    cout << "--- Tạo các loại hộp ---\n";
    HopChu h1;          // Default → delegate → Parameterized
    HopChu h2(4);       // Lập phương → delegate → Parameterized
    HopChu h3(3, 4, 5); // Parameterized trực tiếp
    HopChu h4 = h3;     // Copy

    cout << "\n--- Thể tích ---\n";
    cout << "h1 (1×1×1) = " << h1.the_tich() << "\n";
    cout << "h2 (4×4×4) = " << h2.the_tich() << "\n";
    cout << "h3 (3×4×5) = " << h3.the_tich() << "\n";
    cout << "h4 (copy h3) = " << h4.the_tich() << "\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii2_out vii2_ctor_overload.cpp && ./vii2_out

## VII.3 — Operator Overloading — Tổng quan & Cơ chế

### 🧠 Bản chất: Toán tử là hàm ẩn
Mọi biểu thức `a + b` thực chất là một lời gọi hàm:
```
a + b   →   operator+(a, b)   hoặc   a.operator+(b)
a == b  →   operator==(a, b)  hoặc   a.operator==(b)
cout << x   →   operator<<(cout, x)
```

### 📐 Hai dạng cú pháp

```cpp
// Dạng 1 — Member function: toán hạng TRÁI là *this
ReturnType operator@(const Type& rhs) const { }
//         ↑ gọi là: a.operator@(b)   (a là *this)

// Dạng 2 — Non-member (thường là friend):
friend ReturnType operator@(const Type& lhs, const Type& rhs);
//         ↑ gọi là: operator@(a, b)
```

### ⚖️ Khi nào dùng Member vs Friend?

| Trường hợp | Member | Friend |
|-----------|:------:|:------:|
| `=`, `[]`, `()`, `->` | **Bắt buộc** | ❌ |
| `<<`, `>>` | ❌ | **Bắt buộc** |
| Toán tử số học `+`, `-`, `*` | ✅ Thường dùng | ✅ Nếu cần convert trái |
| Muốn: `3 + myObj` (toán hạng trái khác kiểu) | ❌ Không thể | ✅ Có thể |

> 💡 **Lý do `<<` bắt buộc là friend:**
> `cout << obj` có toán hạng TRÁI là `cout` (kiểu `ostream`), không phải `obj`.
> → Không thể là member của class `obj`.

### 🚫 Toán tử KHÔNG thể nạp chồng
```
::   .   .*   ?:   sizeof   alignof   typeid   noexcept
```


## VII.4 — Toán tử số học (+, -, *, /)

### 🧠 Nguyên tắc thiết kế chuẩn

**Quy tắc 1:** Toán tử số học **return by value** (tạo object mới, không sửa operands)
**Quy tắc 2:** `+=`, `-=` là **member function**, return `*this` (để chain)
**Quy tắc 3:** Xây `+` dựa trên `+=` để tránh lặp code:

```cpp
// ✅ Cách tốt: + tái dùng +=
SoPhuc operator+(const SoPhuc& rhs) const {
    return SoPhuc(*this) += rhs;  // copy *this rồi cộng rhs vào
}
```

### 📌 Class SoPhuc — ví dụ kinh điển
Số phức `a + bi` là bài tập mẫu cho arithmetic overloading:

| Phép toán | Công thức |
|-----------|-----------|
| (a+bi) + (c+di) | (a+c) + (b+d)i |
| (a+bi) - (c+di) | (a-c) + (b-d)i |
| (a+bi) * (c+di) | (ac-bd) + (ad+bc)i |


In [ ]:
%%writefile vii4_arithmetic.cpp
#include <iostream>
using namespace std;

class SoPhuc {
    double re, im;
public:
    SoPhuc(double r = 0, double i = 0) : re(r), im(i) {}

    // ── Compound assignment (member, sửa *this, return *this) ──
    SoPhuc& operator+=(const SoPhuc& rhs) {
        re += rhs.re;  im += rhs.im;
        return *this;
    }
    SoPhuc& operator-=(const SoPhuc& rhs) {
        re -= rhs.re;  im -= rhs.im;
        return *this;
    }
    SoPhuc& operator*=(const SoPhuc& rhs) {
        // (a+bi)(c+di) = (ac-bd) + (ad+bc)i
        double r = re*rhs.re - im*rhs.im;
        double i = re*rhs.im + im*rhs.re;
        re = r;  im = i;
        return *this;
    }

    // ── Arithmetic — TÁI DÙNG compound assignment ──
    SoPhuc operator+(const SoPhuc& rhs) const { return SoPhuc(*this) += rhs; }
    SoPhuc operator-(const SoPhuc& rhs) const { return SoPhuc(*this) -= rhs; }
    SoPhuc operator*(const SoPhuc& rhs) const { return SoPhuc(*this) *= rhs; }

    // ── Unary minus (-a) ──
    SoPhuc operator-() const { return SoPhuc(-re, -im); }

    void in(string label = "") const {
        if (!label.empty()) cout << label << " = ";
        if (im >= 0) cout << re << " + " << im << "i\n";
        else         cout << re << " - " << (-im) << "i\n";
    }
};

int main() {
    SoPhuc a(3, 4);    // 3 + 4i
    SoPhuc b(1, -2);   // 1 - 2i

    a.in("a");
    b.in("b");
    cout << "\n";
    (a + b).in("a+b");
    (a - b).in("a-b");
    (a * b).in("a*b");   // (3+4i)(1-2i) = (3+8)+(−6+4)i = 11−2i
    (-a).in("-a");

    // Chain compound assignment: a += b += c
    SoPhuc c(1, 1);
    cout << "\n--- Chain: a += b += c(1+1i) ---\n";
    a += b += c;   // b+=c trước, rồi a+=b
    a.in("a");  b.in("b");
    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii4_out vii4_arithmetic.cpp && ./vii4_out

## VII.5 — Toán tử so sánh (==, !=, <, >, <=, >=)

### 🧠 Nguyên tắc

**Chỉ implement `==` và `<`** — suy ra 4 toán tử còn lại miễn phí:

```cpp
bool operator!=(const T& r) const { return !(*this == r); }
bool operator> (const T& r) const { return r < *this;     }  // đảo vế
bool operator<=(const T& r) const { return !(r < *this);  }
bool operator>=(const T& r) const { return !(*this < r);  }
```

### 🚀 C++20: Spaceship Operator `<=>`
Chỉ cần 1 dòng, compiler tự sinh ra tất cả 6 toán tử:
```cpp
auto operator<=>(const T& rhs) const = default;
```

### 💡 Lợi ích thực tế: Sau khi có `operator<` → dùng được `std::sort()`!


In [ ]:
%%writefile vii5_comparison.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

class SinhVien {
    string ten;
    double gpa;
public:
    SinhVien(string t, double g) : ten(t), gpa(g) {}

    // ── Chỉ implement 2 toán tử này ──
    bool operator==(const SinhVien& r) const { return gpa == r.gpa; }
    bool operator< (const SinhVien& r) const { return gpa <  r.gpa; }

    // ── 4 toán tử còn lại SUY RA từ 2 cái trên ──
    bool operator!=(const SinhVien& r) const { return !(*this == r); }
    bool operator> (const SinhVien& r) const { return r < *this;     }
    bool operator<=(const SinhVien& r) const { return !(r < *this);  }
    bool operator>=(const SinhVien& r) const { return !(*this < r);  }

    void in() const { cout << ten << "(GPA=" << gpa << ")"; }
    string getTen() const { return ten; }
};

int main() {
    SinhVien a("An",    3.5);
    SinhVien b("Bình",  3.8);
    SinhVien c("Cường", 3.5);

    cout << "a=An(3.5)  b=Bình(3.8)  c=Cường(3.5)\n\n";

    auto tf = [](bool v){ return v ? "✓" : "✗"; };
    cout << "a == c  → " << tf(a==c) << "  (cùng GPA 3.5)\n";
    cout << "a != b  → " << tf(a!=b) << "\n";
    cout << "a <  b  → " << tf(a< b) << "\n";
    cout << "b >  a  → " << tf(b> a) << "\n";
    cout << "a <= c  → " << tf(a<=c) << "\n";
    cout << "b >= a  → " << tf(b>=a) << "\n";

    cout << "\n--- sort() dùng operator< tự động ---\n";
    vector<SinhVien> ds = {b, a, c};
    sort(ds.begin(), ds.end());  // sắp xếp theo GPA tăng dần
    for (auto& sv : ds) { sv.in(); cout << "  "; }
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii5_out vii5_comparison.cpp && ./vii5_out

## VII.6 — Toán tử nhập xuất (<<, >>)

### 🧠 Tại sao BẮT BUỘC là friend (non-member)?

```
cout << obj
    = operator<<(cout, obj)
                 ↑
         toán hạng TRÁI là cout (kiểu ostream)
         → không thể là member của class obj!
         → bắt buộc là non-member function
         → cần friend để truy cập private của obj
```

### 📐 Cú pháp chuẩn — PHẢI nhớ thuộc

```cpp
// Khai báo TRONG class:
friend ostream& operator<<(ostream& os, const MyClass& obj);
friend istream& operator>>(istream& is,       MyClass& obj); // KHÔNG const

// Định nghĩa NGOÀI class:
ostream& operator<<(ostream& os, const MyClass& obj) {
    os << obj.x << " " << obj.y;  // truy cập private được nhờ friend
    return os;   // ← BẮT BUỘC return os để hỗ trợ chain: cout << a << b
}
```

### ⚠️ Lưu ý quan trọng
- `operator>>` nhận rhs là **non-const** (vì phải gán giá trị vào)
- Luôn `return os` / `return is` để hỗ trợ chain: `cout << a << b << c`


In [ ]:
%%writefile vii6_stream.cpp
#include <iostream>
#include <sstream>
using namespace std;

class Vec2D {
    double x, y;
public:
    Vec2D(double x = 0, double y = 0) : x(x), y(y) {}

    // ── Khai báo friend TRONG class ──
    friend ostream& operator<<(ostream& os, const Vec2D& v);
    friend istream& operator>>(istream& is,       Vec2D& v);
};

// ── Định nghĩa NGOÀI class (không viết lại "friend") ──
ostream& operator<<(ostream& os, const Vec2D& v) {
    os << "(" << v.x << ", " << v.y << ")";  // truy cập private v.x, v.y
    return os;   // để chain: cout << a << b
}

istream& operator>>(istream& is, Vec2D& v) {
    is >> v.x >> v.y;  // đọc 2 số
    return is;         // để chain: cin >> a >> b
}

int main() {
    Vec2D a(3, 4), b(1, -2), c(0, 0);

    // Chain <<
    cout << "a = " << a << "\n";
    cout << "b = " << b << "\n";
    cout << "a, b, c = " << a << " | " << b << " | " << c << "\n\n";

    // Đọc từ stringstream (mô phỏng cin >> )
    stringstream ss("7.5 -3.0");
    Vec2D d;
    ss >> d;
    cout << "Đọc từ stream: d = " << d << "\n";

    // Chain >>
    stringstream ss2("1.1 2.2 3.3 4.4");
    Vec2D p, q;
    ss2 >> p >> q;
    cout << "p = " << p << "  q = " << q << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii6_out vii6_stream.cpp && ./vii6_out

## VII.7 — Toán tử gán (=)

### 🧠 Khi nào CẦN viết `operator=`?
Khi class có **tài nguyên động** (con trỏ, file, socket...).
Quy tắc: Nếu cần viết **1 trong 3** thì viết cả 3 (**Rule of Three**):
```
Destructor  +  Copy Constructor  +  Copy Assignment (operator=)
```

### 📐 Cấu trúc chuẩn 4 bước (BẮT BUỘC theo đúng thứ tự!)

```cpp
MyClass& operator=(const MyClass& rhs) {
    // Bước 1: Kiểm tra self-assignment (a = a)
    if (this == &rhs) return *this;

    // Bước 2: Giải phóng tài nguyên CŨ
    delete[] data;

    // Bước 3: Cấp phát mới và DEEP COPY từ rhs
    size = rhs.size;
    data = new int[size];
    copy(rhs.data, rhs.data + size, data);

    // Bước 4: Return *this (để chain: a = b = c)
    return *this;
}
```

### ⚠️ Lỗi kinh điển: Quên bước 1 → Crash khi `a = a`
```cpp
// Nếu không kiểm tra this == &rhs:
delete[] data;        // xóa data của chính mình!
data = new int[size]; // size đã bị hỏng → undefined behavior
```


In [ ]:
%%writefile vii7_assignment.cpp
#include <iostream>
#include <algorithm>  // copy, fill
using namespace std;

class MangSo {
    int* data;
    int  size;
public:
    // Constructor
    MangSo(int n, int val = 0) : size(n), data(new int[n]) {
        fill(data, data + n, val);
        cout << "[Constructor] size=" << n << " val=" << val << "\n";
    }

    // Destructor
    ~MangSo() {
        delete[] data;
        cout << "[Destructor]  size=" << size << "\n";
    }

    // Copy constructor (cần vì có pointer)
    MangSo(const MangSo& o) : size(o.size), data(new int[o.size]) {
        copy(o.data, o.data + size, data);
        cout << "[Copy ctor]\n";
    }

    // ── Copy Assignment Operator ──
    MangSo& operator=(const MangSo& rhs) {
        cout << "[operator=] ";

        // Bước 1: Self-assignment check
        if (this == &rhs) { cout << "self-assign, skip\n"; return *this; }

        // Bước 2: Giải phóng cũ
        delete[] data;

        // Bước 3: Deep copy
        size = rhs.size;
        data = new int[size];
        copy(rhs.data, rhs.data + size, data);

        cout << "done\n";
        return *this;  // Bước 4
    }

    void set(int i, int v) { data[i] = v; }
    int  get(int i) const  { return data[i]; }

    void in(string label = "") const {
        if (!label.empty()) cout << label << " = ";
        cout << "[";
        for (int i = 0; i < size; i++) cout << (i?",":"") << data[i];
        cout << "]\n";
    }
};

int main() {
    MangSo a(3, 1);   // [1,1,1]
    MangSo b(3, 9);   // [9,9,9]

    cout << "\n";
    a.in("a");  b.in("b");

    cout << "\n--- b = a ---\n";
    b = a;
    a.set(0, 99);  // Sửa a[0]

    cout << "\n--- Sau khi sửa a[0]=99 ---\n";
    a.in("a");  b.in("b (không thay đổi — deep copy!)");

    cout << "\n--- Self-assignment: a = a ---\n";
    a = a;  // Phải không crash!

    cout << "\n--- Chain: a = b = MangSo(2,5) ---\n";
    MangSo c(2, 5);   // [5,5]
    a = b = c;        // b=c trước, rồi a=b
    a.in("a");  b.in("b");

    cout << "\n[Destructors khi ra khỏi scope:]\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii7_out vii7_assignment.cpp && ./vii7_out

## VII.8 — Toán tử tăng giảm (++, --)

### 🧠 Cơ chế phân biệt Prefix và Postfix
C++ dùng **tham số giả `int`** để compiler phân biệt hai loại:

```
Prefix  ++a  →  operator++()       không có tham số
Postfix  a++ →  operator++(int)    có tham số int — chỉ là "dấu hiệu"!
```

### 📋 Bảng cú pháp + logic

| Loại | Signature | Return | Logic |
|------|-----------|--------|-------|
| Prefix `++a` | `T& operator++()` | Reference `*this` | **Tăng TRƯỚC**, trả về *mới* |
| Postfix `a++` | `T operator++(int)` | Copy của giá trị *cũ* | Lưu cũ, tăng, trả về *cũ* |

### ⚡ Hiệu năng — Prefix luôn nhanh hơn Postfix!
```
Prefix  ++a:  tăng, return reference   → O(1), KHÔNG tạo copy
Postfix  a++: lưu copy cũ, tăng, return copy → O(1) nhưng TẠO THÊM 1 COPY
```
> 🏆 **Thực tiễn:** Với vòng lặp, **luôn dùng `++i`** thay `i++` (đặc biệt với iterator)

### 📌 Pattern tái sử dụng — Postfix gọi Prefix
```cpp
T operator++(int) {   // Postfix
    T old = *this;    // 1. Lưu bản cũ
    ++(*this);        // 2. Tăng (dùng lại Prefix!)
    return old;       // 3. Trả về bản cũ
}
```


In [ ]:
%%writefile vii8_increment.cpp
#include <iostream>
using namespace std;

class DemSo {
    int gia_tri;
    int buoc;      // bước nhảy mỗi lần tăng/giảm
public:
    DemSo(int v = 0, int b = 1) : gia_tri(v), buoc(b) {}

    // ── PREFIX ++a: tăng rồi trả về *this ──
    DemSo& operator++() {
        gia_tri += buoc;
        return *this;   // trả về bản thân (đã tăng)
    }

    // ── POSTFIX a++: lưu cũ, tăng, trả về cũ ──
    DemSo operator++(int) {    // "int" = dummy, không dùng
        DemSo cu = *this;      // lưu bản cũ
        ++(*this);             // gọi PREFIX để tăng
        return cu;             // trả về bản cũ (by value)
    }

    // ── PREFIX --a ──
    DemSo& operator--() {
        gia_tri -= buoc;
        return *this;
    }

    // ── POSTFIX a-- ──
    DemSo operator--(int) {
        DemSo cu = *this;
        --(*this);
        return cu;
    }

    int  lay() const { return gia_tri; }
    void in(string label = "") const {
        if (!label.empty()) cout << label << "=";
        cout << "DemSo(" << gia_tri << ")";
    }
};

int main() {
    DemSo d(10, 3);   // bắt đầu 10, bước nhảy 3
    cout << "Ban đầu: "; d.in(); cout << "\n\n";

    // ── Prefix ++d: tăng TRƯỚC rồi dùng ──
    DemSo x = ++d;
    cout << "x = ++d: ";
    d.in("d"); cout << "  "; x.in("x(kết quả)");
    cout << "  [x và d cùng giá trị]\n";

    // Reset
    d = DemSo(10, 3);
    cout << "\nReset d=10, buoc=3\n";

    // ── Postfix d++: dùng giá trị hiện tại, RỒIII tăng ──
    DemSo y = d++;
    cout << "y = d++: ";
    d.in("d"); cout << "  "; y.in("y(kết quả)");
    cout << "  [y=10 (giá trị cũ), d=13 (đã tăng)]\n";

    cout << "\n--- Vòng lặp với bước nhảy 2 ---\n";
    DemSo dem(0, 2);
    for (int i = 0; i < 6; ++i, ++dem) {  // dùng ++dem (prefix — nhanh hơn)
        cout << dem.lay() << " ";
    }
    cout << "\n";

    cout << "\n--- Demo -- ---\n";
    DemSo e(20, 5);
    e.in("e"); cout << "  ";
    (--e).in("--e"); cout << "  ";
    (e--).in("e--"); cout << " → "; e.in("e");
    cout << "\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o vii8_out vii8_increment.cpp && ./vii8_out

## VII.9 — Quy tắc nạp chồng toán tử

### 📋 Bảng toán tử và cách triển khai

| Toán tử | Member hay Friend? | Lưu ý |
|---------|:------------------:|-------|
| `=` `[]` `()` `->` | **Bắt buộc Member** | Không thể là non-member |
| `<<` `>>` | **Bắt buộc Friend** | Toán hạng trái là stream |
| `+=` `-=` `*=` `/=` | **Nên Member** | Return `*this` |
| `+` `-` `*` `/` | Thường Member | Non-member nếu cần implicit convert trái |
| `++` `--` prefix | Member | Return `T&` |
| `++` `--` postfix | Member | Return `T` (by value), param `int` dummy |
| `==` `!=` `<` `>` | Member hoặc Friend | Return `bool` |
| `<<` (print) | **Friend** | Return `ostream&` |

### 🚧 Ràng buộc KHÔNG thể phá vỡ

```
1. Arity (số toán hạng): a + b luôn là binary — không đổi được
2. Độ ưu tiên: * luôn cao hơn + — KHÔNG thể đổi
3. Associativity: không thể thay đổi
4. Không thể TẠO toán tử mới: không có ** hay @@
5. Ít nhất 1 toán hạng PHẢI là kiểu user-defined
```

### 🚫 Toán tử KHÔNG thể nạp chồng
```
::   (scope resolution)
.    (member access)
.*   (member pointer dereference)
?:   (ternary conditional)
sizeof   alignof   typeid   noexcept
```

### 💡 Nguyên tắc Vàng
```
✅ Hành vi phải TRỰC QUAN — khớp với kỳ vọng người đọc code
   (Principle of Least Astonishment)

✅ Nếu overload ==  → overload != luôn
✅ Nếu overload <   → overload tất cả 6 toán tử so sánh
✅ operator= phải là member
✅ operator<< phải là friend

❌ KHÔNG overload && và || → mất short-circuit evaluation!
❌ KHÔNG overload , (comma operator)
❌ KHÔNG dùng + để làm phép trừ (phá vỡ tính trực quan)
```


---
# VIII. FRIEND TRONG C++

> **Friend** = "Người bạn được tin tưởng" — được phép vào **khu vực riêng tư**.
> Cơ chế cho phép **phá vỡ encapsulation có kiểm soát**.

```
         ┌──────────────────────────────────┐
         │           class A                │
         │  private:                        │
         │    int x;  ◄─────── Friend truy cập được!
         │    int y;  ◄─────── Friend truy cập được!
         │  public:                         │
         │    friend void f(A&);  ← khai báo│
         │    friend class B;     ← khai báo│
         └──────────────────────────────────┘
```

| Loại | Cú pháp trong class | Phạm vi quyền |
|------|--------------------|-|
| Friend Function | `friend ReturnType funcName(params);` | Chỉ hàm đó |
| Friend Class | `friend class ClassName;` | Toàn bộ class đó |


## VIII.1 — Friend Function (Hàm bạn)

### 🧠 Đặc điểm then chốt

| Đặc điểm | Giải thích |
|----------|-----------|
| Không phải member | Không có `this`, không gọi qua `obj.method()` |
| Khai báo trong class | Dùng từ khóa `friend` — có thể ở `public` hoặc `private`, không quan trọng |
| Định nghĩa ngoài class | **Không** có `ClassName::`, **không** có `friend` lại |
| Không kế thừa | Subclass KHÔNG tự động có friend function của parent |

### 📐 Cú pháp — 3 bước

```cpp
// BƯỚC 1 — Khai báo TRONG class:
class A {
    int bi_mat;
    friend void ham_ban(const A& a);   // khai báo friend
};

// BƯỚC 2 — Định nghĩa NGOÀI class:
void ham_ban(const A& a) {   // không có A:: , không có friend
    cout << a.bi_mat;         // truy cập private được!
}

// BƯỚC 3 — Gọi như hàm thông thường:
A obj;
ham_ban(obj);   // KHÔNG phải obj.ham_ban()
```


In [ ]:
%%writefile viii1_friend_func.cpp
#include <iostream>
#include <cmath>
using namespace std;

class Diem {
    double x, y;
public:
    Diem(double x = 0, double y = 0) : x(x), y(y) {}

    // ── Khai báo friend functions ──
    friend double khoang_cach(const Diem& a, const Diem& b);
    friend ostream& operator<<(ostream& os, const Diem& d);
    friend Diem giua(const Diem& a, const Diem& b);  // điểm giữa

    // ── Member function để so sánh ──
    double den_goc() const { return sqrt(x*x + y*y); }
};

// ── Định nghĩa ngoài class — KHÔNG có Diem:: ──
double khoang_cach(const Diem& a, const Diem& b) {
    return sqrt((a.x-b.x)*(a.x-b.x) + (a.y-b.y)*(a.y-b.y));
    // Truy cập a.x, b.x (private) được nhờ friend!
}

Diem giua(const Diem& a, const Diem& b) {
    return Diem((a.x+b.x)/2, (a.y+b.y)/2);
}

ostream& operator<<(ostream& os, const Diem& d) {
    os << "(" << d.x << ", " << d.y << ")";
    return os;
}

int main() {
    Diem p1(0, 0), p2(3, 4), p3(6, 8);

    cout << "p1 = " << p1 << "\n";
    cout << "p2 = " << p2 << "\n";
    cout << "p3 = " << p3 << "\n\n";

    // Gọi friend function như hàm thường (KHÔNG phải p1.khoang_cach())
    cout << "Khoảng cách p1-p2: " << khoang_cach(p1, p2) << "\n";
    cout << "Khoảng cách p2-p3: " << khoang_cach(p2, p3) << "\n";
    cout << "Điểm giữa p1-p3:   " << giua(p1, p3)        << "\n";

    // So sánh với member function
    cout << "\nMember vs Friend:\n";
    cout << "p2.den_goc() [member] = " << p2.den_goc()         << "\n";
    cout << "khoang_cach(p1,p2) [friend] = " << khoang_cach(p1,p2) << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -o viii1_out viii1_friend_func.cpp && ./viii1_out

## VIII.2 — Friend Class (Lớp bạn)

### 🧠 Đặc điểm then chốt

Khi B là friend của A: **TẤT CẢ** member function của B truy cập được private/protected của A.

### ⚠️ Ba điều BẮT BUỘC phải nhớ

```
1. KHÔNG tương xứng (Asymmetric):
       B là friend của A  ≠  A là friend của B

2. KHÔNG bắc cầu (Non-transitive):
       B friend A, C friend B  ≠  C friend A

3. KHÔNG kế thừa (Non-inheritable):
       B là friend của A  ≠  subclass của B cũng là friend của A
```

### 📐 Cú pháp
```cpp
class A {
    int bi_mat;
    friend class B;   // B có thể truy cập TẤT CẢ private của A
};

class B {
    void method(A& a) {
        a.bi_mat = 42;  // ✅ Được!
    }
};
```


In [ ]:
%%writefile viii2_friend_class.cpp
#include <iostream>
using namespace std;

class DongCo {
    double cong_suat;    // private!
    int    so_vong;      // private! RPM
    bool   dang_chay;    // private!

    friend class Xe;   // Xe có thể truy cập TẤT CẢ private của DongCo

public:
    DongCo(double cs, int sv) : cong_suat(cs), so_vong(sv), dang_chay(false) {}

    void in() const {
        cout << "  Engine: " << cong_suat << "HP, "
             << so_vong << " RPM, "
             << (dang_chay ? "running" : "stopped") << "\n";
    }
};

class Xe {
    string  ten;
    DongCo  dongco;
public:
    Xe(string t, double cs, int sv) : ten(t), dongco(cs, sv) {}

    void khoi_dong() {
        // Truy cập private của DongCo được nhờ friend class!
        dongco.dang_chay = true;
        dongco.so_vong   = 800;
        cout << "[" << ten << "] Khởi động — idle RPM=800\n";
    }

    void tang_toc(int muc) {
        if (!dongco.dang_chay) {
            cout << "[" << ten << "] Chưa khởi động!\n"; return;
        }
        dongco.so_vong += muc * 200;
        cout << "[" << ten << "] Tăng tốc × " << muc
             << " → RPM=" << dongco.so_vong << "\n";
    }

    void tat_may() {
        dongco.dang_chay = false;
        dongco.so_vong   = 0;
        cout << "[" << ten << "] Tắt máy\n";
    }

    void in() const { cout << "Xe: " << ten << "\n"; dongco.in(); }
};

int main() {
    Xe xe("Toyota Yaris", 97, 6000);
    xe.in();

    cout << "\n";
    xe.khoi_dong();
    xe.tang_toc(2);
    xe.tang_toc(3);
    xe.in();

    cout << "\n";
    xe.tat_may();
    xe.in();

    // ── Minh họa: KHÔNG tương xứng ──
    // Từ bên ngoài, vẫn KHÔNG thể truy cập private của DongCo!
    // DongCo dc(100, 3000);
    // dc.cong_suat = 200;  // ❌ error: 'cong_suat' is private

    cout << "\nLưu ý: Từ bên ngoài class Xe vẫn KHÔNG thể access private của DongCo!\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o viii2_out viii2_friend_class.cpp && ./viii2_out

## VIII.3 — Ứng dụng Friend trong thực tế

### 🎯 Các tình huống dùng Friend phổ biến

| Tình huống | Loại Friend | Ví dụ |
|-----------|:-----------:|-------|
| Operator `<<`, `>>` | Friend Function | Bắt buộc! |
| Hai class phối hợp chặt | Friend Class | Bank+Account, Engine+Car |
| Toán tử cần 2 objects | Friend Function | Matrix × Vector |
| Iterator + Container | Friend Class | `list` + `list::iterator` trong STL |
| Unit Testing | Friend Class | Test class cần xem internal state |

### 🏦 Ví dụ: Ngân hàng + Tài khoản
Hệ thống ngân hàng: `NganHang` cần đọc/ghi số dư `TaiKhoan` — thông tin private.


In [ ]:
%%writefile viii3_applications.cpp
#include <iostream>
#include <vector>
#include <iomanip>
using namespace std;

// ── Forward declaration ──
class NganHang;

class TaiKhoan {
    string ten_chu;
    double so_du;           // private!
    vector<string> lich_su; // lịch sử giao dịch — private!

    friend class NganHang;  // NganHang có thể đọc/ghi private của TaiKhoan

public:
    TaiKhoan(string ten, double so_du_ban_dau)
        : ten_chu(ten), so_du(so_du_ban_dau) {
        lich_su.push_back("Mở tài khoản: " + to_string((int)so_du_ban_dau));
    }

    // Public: chủ tài khoản chỉ được XEM — không được SỬA trực tiếp
    string ten()     const { return ten_chu; }
    double soDu()    const { return so_du; }
    void   inBaoCao() const {
        cout << "┌─ Tài khoản: " << ten_chu << "\n";
        cout << "│  Số dư: " << fixed << setprecision(0) << so_du << " VND\n";
        cout << "└─ Lịch sử:\n";
        for (auto& s : lich_su) cout << "   • " << s << "\n";
    }
};

class NganHang {
    string ten_ngan_hang;
public:
    NganHang(string t) : ten_ngan_hang(t) {}

    // ── Nghiệp vụ: chuyển khoản ──
    bool chuyen_khoan(TaiKhoan& tu, TaiKhoan& den, double so_tien) {
        // Truy cập private của TaiKhoan được nhờ friend class!
        if (tu.so_du < so_tien) {
            cout << "[" << ten_ngan_hang << "] Số dư không đủ!\n";
            return false;
        }
        tu.so_du  -= so_tien;   // truy cập private!
        den.so_du += so_tien;   // truy cập private!

        string note = "CK " + to_string((int)so_tien) + " VND đến " + den.ten();
        tu.lich_su.push_back(note);                      // truy cập private!
        den.lich_su.push_back("Nhận " + to_string((int)so_tien) + " từ " + tu.ten());

        cout << "[" << ten_ngan_hang << "] Chuyển khoản OK: "
             << tu.ten() << " → " << den.ten()
             << " | " << (int)so_tien << " VND\n";
        return true;
    }

    // ── Kiểm tra tài khoản (nghiệp vụ nội bộ) ──
    void kiem_tra(const TaiKhoan& tk) {
        cout << "[Kiểm tra nội bộ] " << tk.ten_chu
             << " — Số dư: " << tk.so_du << "\n";  // truy cập private!
    }
};

int main() {
    TaiKhoan an("An",     5000000);
    TaiKhoan binh("Bình", 2000000);
    NganHang vcb("Vietcombank");

    cout << "=== Trạng thái ban đầu ===\n";
    an.inBaoCao();
    binh.inBaoCao();

    cout << "\n=== Thực hiện giao dịch ===\n";
    vcb.chuyen_khoan(an, binh, 1500000);
    vcb.chuyen_khoan(binh, an, 999999999);  // sẽ thất bại
    vcb.chuyen_khoan(binh, an, 500000);

    cout << "\n=== Trạng thái sau giao dịch ===\n";
    an.inBaoCao();
    cout << "\n";
    binh.inBaoCao();
    return 0;
}


In [ ]:
!g++ -std=c++17 -o viii3_out viii3_applications.cpp && ./viii3_out

## VIII.4 — Ưu nhược điểm của Friend

### ✅ Ưu điểm

| Ưu điểm | Giải thích |
|---------|-----------|
| **Bắt buộc cho `<<`, `>>`** | Không có cách nào khác! |
| **Hiệu năng** | Truy cập trực tiếp, không qua getter/setter → zero overhead |
| **Hỗ trợ symmetric operations** | `3 + myObj` thay vì chỉ `myObj + 3` |
| **Phối hợp class chặt chẽ** | Iterator+Container, Engine+Car — thiết kế tự nhiên |

### ❌ Nhược điểm

| Nhược điểm | Giải thích |
|-----------|-----------|
| **Phá vỡ Encapsulation** | `private` không còn thực sự riêng tư |
| **Tăng Coupling** | Class A phải biết tên của B |
| **Khó maintain** | Sửa private của A → có thể ảnh hưởng tất cả friend |
| **Không kế thừa** | Subclass không tự động được friend |

### 💡 Quyết định sử dụng Friend — Cây quyết định

```
Cần truy cập private của class khác?
│
├── Có thể dùng public getter/setter? ──→ Dùng getter/setter (OOP cleaner)
│
├── Toán tử là <<, >> ? ──────────────→ Dùng Friend (bắt buộc)
│
├── Hai class phải phối hợp CỰC KỲ chặt?
│   (Iterator+Container, Matrix+Vec) ──→ Dùng Friend Class
│
└── Chỉ muốn "tiện" tránh viết getter? ─→ KHÔNG dùng Friend
```

### 🎯 Tóm tắt thực chiến
```
Ưu tiên:
  1. Member function (sạch nhất — OOP thuần túy)
  2. Getter/Setter (encapsulation tốt)
  3. Friend function/class (khi thực sự cần — có lý do rõ ràng)
```


---
## ★ BÀI TỔNG HỢP — Class PhanSo (Phân số)

### Mục tiêu: Áp dụng TẤT CẢ kiến thức VII + VIII vào một class

Class `PhanSo` tích hợp:
- ✅ **Constructor overloading** (default, parameterized, copy)
- ✅ **Toán tử số học** (+, -, *, /) — tái dùng +=
- ✅ **Toán tử so sánh** (==, !=, <, >=)
- ✅ **Toán tử nhập xuất** (<< là friend)
- ✅ **Toán tử gán** (=)
- ✅ **Tự động rút gọn** phân số (dùng GCD)

```
         tu        1
PhanSo = ──   →   ───
         mau       3
```


In [ ]:
%%writefile capstone_phanso.cpp
#include <iostream>
#include <stdexcept>
using namespace std;

class PhanSo {
    int tu, mau;   // tử số, mẫu số

    // ── Hàm riêng: Ước chung lớn nhất (Euclidean algorithm) ──
    static int gcd(int a, int b) {
        return b == 0 ? a : gcd(b, a % b);
    }

    // ── Hàm riêng: Rút gọn phân số ──
    void don_gian() {
        if (mau == 0) throw invalid_argument("Mẫu số không được = 0!");
        if (mau < 0) { tu = -tu; mau = -mau; }  // mẫu luôn dương
        int g = gcd(abs(tu), abs(mau));
        tu /= g;  mau /= g;
    }

public:
    // ─────────────────────────────────────────
    // CONSTRUCTOR OVERLOADING
    // ─────────────────────────────────────────
    PhanSo()                       : tu(0), mau(1) {}         // 0/1
    PhanSo(int t)                  : tu(t), mau(1) {}         // t/1
    PhanSo(int t, int m)           : tu(t), mau(m) { don_gian(); }
    PhanSo(const PhanSo& o)        : tu(o.tu), mau(o.mau)  {} // copy

    // ─────────────────────────────────────────
    // ASSIGNMENT OPERATOR
    // ─────────────────────────────────────────
    PhanSo& operator=(const PhanSo& rhs) {
        if (this != &rhs) { tu = rhs.tu; mau = rhs.mau; }
        return *this;
    }

    // ─────────────────────────────────────────
    // ARITHMETIC — COMPOUND (+=, -=, *=, /=)
    // ─────────────────────────────────────────
    PhanSo& operator+=(const PhanSo& r) {
        tu = tu*r.mau + r.tu*mau;
        mau = mau * r.mau;
        don_gian();  return *this;
    }
    PhanSo& operator-=(const PhanSo& r) {
        tu = tu*r.mau - r.tu*mau;
        mau = mau * r.mau;
        don_gian();  return *this;
    }
    PhanSo& operator*=(const PhanSo& r) {
        tu *= r.tu;  mau *= r.mau;
        don_gian();  return *this;
    }
    PhanSo& operator/=(const PhanSo& r) {
        tu *= r.mau;  mau *= r.tu;  // nhân nghịch đảo
        don_gian();  return *this;
    }

    // ─────────────────────────────────────────
    // ARITHMETIC — TÁI DÙNG compound
    // ─────────────────────────────────────────
    PhanSo operator+(const PhanSo& r) const { return PhanSo(*this) += r; }
    PhanSo operator-(const PhanSo& r) const { return PhanSo(*this) -= r; }
    PhanSo operator*(const PhanSo& r) const { return PhanSo(*this) *= r; }
    PhanSo operator/(const PhanSo& r) const { return PhanSo(*this) /= r; }
    PhanSo operator-() const { return PhanSo(-tu, mau); }

    // ─────────────────────────────────────────
    // COMPARISON
    // ─────────────────────────────────────────
    bool operator==(const PhanSo& r) const { return tu*r.mau == r.tu*mau; }
    bool operator< (const PhanSo& r) const { return tu*r.mau <  r.tu*mau; }
    bool operator!=(const PhanSo& r) const { return !(*this == r); }
    bool operator<=(const PhanSo& r) const { return !(*this >  r); }
    bool operator> (const PhanSo& r) const { return r < *this;     }
    bool operator>=(const PhanSo& r) const { return !(*this <  r); }

    // ─────────────────────────────────────────
    // STREAM — friend (bắt buộc)
    // ─────────────────────────────────────────
    friend ostream& operator<<(ostream& os, const PhanSo& p);
    friend istream& operator>>(istream& is, PhanSo& p);

    double toDouble() const { return (double)tu / mau; }
};

ostream& operator<<(ostream& os, const PhanSo& p) {
    if (p.mau == 1) os << p.tu;          // số nguyên
    else            os << p.tu << "/" << p.mau;
    return os;
}
istream& operator>>(istream& is, PhanSo& p) {
    char slash;
    is >> p.tu >> slash >> p.mau;  // định dạng "3/4"
    p.don_gian();
    return is;
}

int main() {
    PhanSo a(1, 2);    // 1/2
    PhanSo b(1, 3);    // 1/3
    PhanSo c(3, 6);    // 3/6 → tự rút gọn thành 1/2
    PhanSo d(5);       // 5/1 = 5

    cout << "=== Khởi tạo ===\n";
    cout << "a = " << a << "  (" << a.toDouble() << ")\n";
    cout << "b = " << b << "  (" << b.toDouble() << ")\n";
    cout << "c = " << c << "  (3/6 tự rút gọn!)\n";
    cout << "d = " << d << "\n";

    cout << "\n=== Phép tính ===\n";
    cout << a << " + " << b << " = " << (a + b) << "\n";  // 1/2+1/3 = 5/6
    cout << a << " - " << b << " = " << (a - b) << "\n";  // 1/2-1/3 = 1/6
    cout << a << " * " << b << " = " << (a * b) << "\n";  // 1/2*1/3 = 1/6
    cout << a << " / " << b << " = " << (a / b) << "\n";  // 1/2÷1/3 = 3/2

    cout << "\n=== So sánh ===\n";
    cout << a << " == " << c << " → " << (a == c ? "true" : "false") << "\n";
    cout << a << " >  " << b << " → " << (a >  b ? "true" : "false") << "\n";
    cout << b << " <  " << a << " → " << (b <  a ? "true" : "false") << "\n";

    cout << "\n=== Chain phép tính ===\n";
    PhanSo tong;
    tong += PhanSo(1,2);
    tong += PhanSo(1,3);
    tong += PhanSo(1,6);
    cout << "1/2 + 1/3 + 1/6 = " << tong << "\n";  // = 1

    return 0;
}


In [ ]:
!g++ -std=c++17 -o capstone_out capstone_phanso.cpp && ./capstone_out

---
# 📝 TỔNG KẾT — CHEAT SHEET

## Operator Overloading

```cpp
// ── ARITHMETIC ──────────────────────────────────────────
T& operator+=(const T& r);                  // member, return *this
T  operator+(const T& r) const              // tái dùng +=
    { return T(*this) += r; }

// ── COMPARISON ──────────────────────────────────────────
bool operator==(const T& r) const;          // implement thực sự
bool operator< (const T& r) const;          // implement thực sự
bool operator!=(const T& r) const { return !(*this == r); }  // suy ra
bool operator> (const T& r) const { return r < *this;    }  // suy ra

// ── STREAM (BẮT BUỘC FRIEND) ─────────────────────────────
friend ostream& operator<<(ostream& os, const T& obj);    // return os
friend istream& operator>>(istream& is,       T& obj);    // return is

// ── ASSIGNMENT ───────────────────────────────────────────
T& operator=(const T& r) {
    if (this == &r) return *this;   // 1. self-check
    delete[] data;                  // 2. giải phóng cũ
    /* deep copy */                 // 3. copy mới
    return *this;                   // 4. return *this
}

// ── PREFIX / POSTFIX ─────────────────────────────────────
T& operator++()     { /*tăng*/; return *this; }  // prefix
T  operator++(int)  { T old=*this; ++(*this); return old; }  // postfix
```

## Friend

```cpp
class A {
    int x;
    friend void ham_ban(A&);      // friend function — chỉ hàm đó
    friend class B;               // friend class — toàn bộ B
    friend ostream& operator<<(ostream&, const A&);  // stream operator
};
// Định nghĩa ngoài: void ham_ban(A& a) { a.x = 1; }  // KHÔNG có friend, KHÔNG có A::
```

## Bảng quyết định

```
Toán tử = [] () ->         →  Bắt buộc Member
Toán tử << >>              →  Bắt buộc Friend (non-member)
Toán tử += -= *= /=        →  Nên Member, return *this
Toán tử + - * /            →  Member (tái dùng +=)
Prefix/Postfix ++ --        →  Member (phân biệt bằng dummy int)
Toán tử so sánh            →  Chỉ implement == và <, suy ra 4 cái còn lại
```

## Top 5 lỗi hay gặp

```cpp
// ❌ 1. Quên return *this
T& operator=(const T& r) { /* ... */ }           // thiếu return!

// ❌ 2. Quên self-assignment check
delete[] data; data = new int[r.size];           // crash nếu this == &r

// ❌ 3. Postfix trả về mới thay vì cũ
T operator++(int) { return ++(*this); }          // SAI — phải return old

// ❌ 4. Stream operator là member
ostream& operator<<(ostream& os) { ... }         // SAI — không có os làm this

// ❌ 5. Return type không phải signature
int  tinh(int a) { return a; }
void tinh(int a) { }                             // SAI — trùng signature!
```
